# 选择器

学习目标：能按元素身份、文档关系和交互状态选择目标，并区分没有匹配与选择器无效。

前置知识：HTML 元素、class/id 属性、父子与兄弟关系；CSS 规则、声明和外部样式表。

适用范围：基础选择器与 Selectors Level 4 中已实现的常用功能；Level 4 仍按工作草案演进，:has()等功能须核对目标浏览器。页面无需 JavaScript。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/02-selectors/。

1. [index.html](scripts/02-selectors/index.html)、[index.css](scripts/02-selectors/index.css)：基本匹配、组合器、结构与函数伪类及伪元素。
2. [states.html](scripts/02-selectors/states.html)、[states.css](scripts/02-selectors/states.css)：表单状态、鼠标与键盘焦点对照。

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/02-selectors/index.html)。

保存修改后刷新页面。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 类型、类、ID 与通配选择器

先用不同标记选中一段通知。p 选择 &lt;p&gt; 元素；.notice 选择 class 中包含 notice 的元素；#headline 选择 id 为 headline 的元素。点和井号属于 CSS 语法，不写入 HTML 的属性值。

- class 可包含多个用空白分隔的类名。.notice.urgent 要求同一元素同时有这两个类，HTML 中类名排列次序不影响匹配。
- p.notice 同时限定元素类型与类名。.notice p 中的空格则表达后代关系，后面继续区分。
- * 是通配选择器，匹配元素，不会直接选择文本节点；.basic * 把范围限定为 .basic 内的后代。
- id 在同一文档树中应唯一。类名和 ID 的大小写要与 HTML 对应；本章选用字母开头、连字符分词的名字，避免数字开头等名称需要 CSS 转义的情况。

本章用 color（文字颜色）、background-color（背景颜色）、border（边框）、outline（轮廓）等属性标出匹配对象；它们不是选择器。

```html
<div class="basic">
  <p id="headline" class="notice urgent">紧急通知</p>
  <p class="notice">普通通知</p>
  <span class="notice">行内通知</span>
</div>
```

```css
.basic * { outline: 1px dotted gray; }
.basic p { color: navy; }
.notice { background-color: lightyellow; }
.notice.urgent { border: 2px solid maroon; }
#headline { font-size: 20px; }
/* 检查：三个后代都有轮廓和背景；紧急通知额外有边框与 20px 字号。 */
```

配套文件：[index.html](scripts/02-selectors/index.html)、[index.css](scripts/02-selectors/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/02-selectors/index.html#demo-1)

## 2 属性选择器检查 HTML 属性

方括号中的名称是 HTML 属性，不是 CSS 属性。例如 [data-state] 检查元素是否写了 data-state；[data-state="ready"] 要求整个属性值相等。

- [data-tags~="css"] 检查空白分隔的词中是否有 css；不会把 css-course 当成 css。
- [lang|="zh"] 匹配 zh 或以 zh- 开头的值，适合这种语言代码写法；不会读取未写出的 lang 属性。
- [data-file^="lesson-"]、[data-file$=".css"]、[data-file*="core"] 分别检查前缀、后缀、包含的子串。
- [data-kind="note" i] 中 i 要求 ASCII 字母不区分大小写。默认是否区分与文档及属性有关，class、id、data-* 的值通常区分大小写。s 是强制区分标记，但支持情况有差异，本例不依赖它。

这些比较处理属性里的字符串，不会推断文件真正的类型。示例用 data-file 保存文件名文本，不发起文件请求。

```html
<p class="attribute-demo" data-state="ready" data-tags="css core"
   data-file="lesson-core.css" data-kind="NOTE" lang="zh-CN">CSS 文件记录</p>
<p class="attribute-demo" data-state="waiting" data-tags="css-course"
   data-file="draft.txt">待整理记录</p>
```

```css
.attribute-demo[data-state] { border: 1px solid gray; }
.attribute-demo[data-state="ready"] { color: navy; }
.attribute-demo[data-tags~="css"] { background-color: lightyellow; }
.attribute-demo[lang|="zh"] { font-weight: bold; }
.attribute-demo[data-file^="lesson-"][data-file$=".css"] { border-color: teal; }
.attribute-demo[data-file*="core"] { border-width: 3px; }
.attribute-demo[data-kind="note" i] { outline: 2px dashed purple; }
/* 检查第二条：css-course 不匹配完整词 css，因此没有浅黄色背景。 */
```

配套文件：[index.html](scripts/02-selectors/index.html)、[index.css](scripts/02-selectors/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/02-selectors/index.html#demo-2)

## 3 组合器与匹配范围

先画出父子与兄弟关系，再读组合器。本例 .relations 是演示容器的类名；选择器最终给最右侧匹配的元素设置样式，不能只按源码中相邻的几行判断。

- .relations p：容器中任意深度的 &lt;p&gt; 后代。
- .relations > p：容器的直接 &lt;p&gt; 子元素，不包含更深一层的段落。
- h3 + p：紧跟在 &lt;h3&gt; 后面的 &lt;p&gt; 元素兄弟，中间的空白和注释不算元素。
- h3 ~ p：同一父元素下，出现在 &lt;h3&gt; 之后的所有 &lt;p&gt; 兄弟，不要求紧邻。

普通的后代、子代和兄弟组合器不会直接选中左侧的祖先或前面的兄弟。限制到容器只限制直接匹配范围；文字颜色等可继承属性还可能传给目标的子元素，继承在下一章展开。

![relations 下依次是 h3、direct-one、包含 nested-one 的 div 和 direct-two；四种组合器分别选择不同段落。](image/illustration/02-01-selector-tree.svg)

图 1：下方 HTML 的元素子树。兄弟按源码顺序从左向右排，空白文本与注释不参与这里的元素关系判断。

把四条 CSS 分别落到图中的 p 节点，再对照边框、背景与粗体；嵌套段落有颜色却没有直接子代边框。

```html
<div class="relations">
  <h3>阅读安排</h3>
  <p id="direct-one">直接子段落一</p>
  <div><p id="nested-one">嵌套段落</p></div>
  <p id="direct-two">直接子段落二</p>
</div>
<p id="outside-one">容器外的段落</p>
```

```css
.relations p { color: navy; }
.relations > p { border: 2px solid teal; }
.relations h3 + p { background-color: lightyellow; }
.relations h3 ~ p { font-weight: bold; }
/* 检查：嵌套段落只有蓝色；第二个直接子段落有边框、粗体，但没有黄色背景。 */
```

配套文件：[index.html](scripts/02-selectors/index.html)、[index.css](scripts/02-selectors/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/02-selectors/index.html#demo-3)

## 4 选择器列表与整条规则失效

逗号将多个选择器并列，任意一项匹配就应用声明。逗号两边各是完整选择器；.list-demo h3, p 的第二项会选择整个文档的 p，不能把前面的容器限制自动延续过去。

普通选择器列表中，一项语法无效或不受支持，通常会使整条规则被忽略。下面的 :unknown-state 是故意写出的未知伪类。分开写的其他有效规则仍可生效。

:is() 和 :where() 的参数采用容错选择器列表（forgiving selector list），可以忽略其中无效的分支；不要把这条规则套到 :not() 或 :has() 参数上。

```html
<div class="list-demo">
  <h3>阅读列表</h3>
  <p class="list-target">这一段用来比较有效与无效列表。</p>
</div>
```

```css
.list-demo h3, .list-demo p { color: navy; }
.list-target { border: 2px solid teal; }
/* 反例：未知伪类使下面整条规则无效，边框不会变成红色。 */
.list-target, :unknown-state { border-color: red; }
:is(.list-target, :unknown-state) { background-color: lightyellow; }
/* 支持 :is() 的浏览器保留有效分支，段落有黄色背景。 */
```

配套文件：[index.html](scripts/02-selectors/index.html)、[index.css](scripts/02-selectors/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/02-selectors/index.html#demo-4)

## 5 状态伪类与键盘交互

伪类（pseudo-class）用单冒号，为已有元素增加匹配条件。本页的控件使用原生 HTML 行为；CSS 只改变呈现。

- :hover 表示指针悬停；触屏未必提供稳定的悬停状态，必要操作不能只靠它出现。
- :active 对应正在激活的短暂阶段，如按住按钮；它不表示“当前选中的菜单项”。
- :focus 匹配当前焦点元素；:focus-visible 还要求浏览器判断应显示焦点提示，不能简单等同于“只由键盘触发”。:focus-within 也匹配含焦点后代的容器。
- :checked 跟随复选框、单选按钮等当前选中状态；[checked] 检查的是 HTML 属性，用户操作后两者不一定相同。:disabled/:enabled 按控件禁用语义匹配，不是任意元素加同名类。
- :required、:invalid 可标出必填与不满足约束的输入框；必填空值可能在页面刚打开时就无效，正式表单还需考虑反馈时机。

链接的 :link/:visited 区分未访问与已访问状态，后者只能改变受限的颜色类属性，而且计算样式会隐藏访问历史。不要用脚本读颜色断言访问记录。

```html
<div class="state-panel">
  <label>姓名 <input id="person-name" required></label>
  <label><input id="agree" type="checkbox"><span id="agree-label">接收阅读提醒</span></label>
  <button id="state-button" type="button">按住观察</button>
  <button type="button" disabled>暂不可用</button>
  <a href="index.html#demo-1">返回本章第一组</a>
</div>
```

```css
.state-panel { border: 2px solid gray; padding: 12px; }
.state-panel:focus-within { border-color: teal; }
.state-panel input:focus { outline: 2px solid navy; }
.state-panel button:focus-visible { outline: 3px solid navy; }
.state-panel button:hover { background-color: lightyellow; }
.state-panel button:active { background-color: gold; }
.state-panel button:disabled { border-style: dashed; }
.state-panel input:required:invalid { border: 2px solid maroon; }
#agree:checked + span { font-weight: bold; }
.state-panel a:link { color: navy; }
.state-panel a:visited { color: purple; }
/* 用 Tab 移动焦点、空格切换复选框；保留浏览器原生焦点提示作为回退。 */
```

配套文件：[states.html](scripts/02-selectors/states.html)、[states.css](scripts/02-selectors/states.css) · [浏览器预览](http://127.0.0.1:8101/scripts/02-selectors/states.html#demo-5)

## 6 结构伪类按兄弟位置计数

结构伪类按元素在文档树中的位置匹配，不按视觉排版位置计算。计数从 1 开始，空白文本与注释不占元素序号。

- :first-child、:last-child 分别要求是第一个、最后一个元素子节点；:only-child 要求没有其他元素兄弟。
- p:nth-child(2) 要求元素是 p，且在所有类型的元素兄弟中排第 2。
- p:nth-of-type(2) 则只在同类型 p 兄弟中排第 2。
- :nth-child(2n+1) 选择第 1、3、5…个，等同 odd；n 依次取 0、1、2…，2 是步长，1 是偏移。even 等同 2n。负步长如 -n+3 可选前三个。
- :nth-last-child() 从最后一个元素倒数，:first-of-type/:last-of-type 按类型选择首尾。

带 of 的扩展写法可先过滤再计数，如 :nth-child(2 of .entry)；它的支持范围不同于基础形式，本章示例使用基础形式，旧浏览器需要固定类名时可以直接标记目标。

```html
<div class="structure-demo">
  <h3>本周</h3>
  <p id="struct-a">第一段，也是第二个元素子节点</p>
  <p id="struct-b">第二段，也是第三个元素子节点</p>
  <p id="struct-c">第三段，也是第四个元素子节点</p>
</div>
<div class="single-demo"><p>唯一的元素子节点</p></div>
```

```css
.structure-demo > p:first-child { color: red; }
.structure-demo > p:nth-child(2) { color: navy; }
.structure-demo > p:nth-of-type(2) { background-color: lightyellow; }
.structure-demo > p:nth-child(2n+1) { border: 2px solid teal; }
.structure-demo > p:last-child { font-weight: bold; }
.single-demo > p:only-child { outline: 1px dashed teal; }
/* 检查：第一段不是 first-child；第二段有背景和边框；第三段为粗体。 */
```

配套文件：[index.html](scripts/02-selectors/index.html)、[index.css](scripts/02-selectors/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/02-selectors/index.html#demo-6)

## 7 :not()、:is()、:where() 与 :has()

这些函数形式仍是伪类，参数写选择器。

- .choice:not(.muted) 排除自身有 muted 类的候选；它不会自动表示“排除 muted 祖先中的全部后代”。
- .choice:is(.book, .video) 允许自身属于两个分支中的任意一个。
- :where() 与 :is() 的匹配方式相同，但 :where() 连同参数的选择器优先级为零；其他部分仍照常计数。下一章解释覆盖顺序。
- .relational-card:has(> input:checked) 选择直接子复选框已勾选的卡片；:has(+ p) 这类参数也能根据后续兄弟匹配当前元素，因此不能只把它记成“父选择器”。

:has() 不能嵌套 :has()，本章所用语法也不允许把伪元素放进参数。先保留可操作的原生复选框，再把卡片着色作为增强；@supports selector(...) 只在浏览器支持该选择器语法时应用内部规则，它是 @ 规则，不是选择器。

MDN 将 :has() 列为自 2023 年 12 月起进入跨浏览器可用范围；这不涵盖所有旧版本。下面的相邻兄弟提示是独立回退，不依赖 :has()。

```html
<p class="choice book">书籍</p>
<p class="choice video muted">视频（暂缓）</p>
<div class="relational-card">
  <input id="card-check" type="checkbox"><label for="card-check">加入阅读计划</label>
</div>
```

```css
.choice:not(.muted) { color: navy; }
.choice:is(.book, .video) { border: 1px solid teal; }
:where(.choice) { background-color: lightyellow; }
.relational-card { border: 2px solid gray; padding: 12px; }
.relational-card > input:checked + label { font-weight: bold; }
@supports selector(:has(*)) {
  .relational-card:has(> input:checked) { border-color: teal; }
}
/* 勾选后标签变粗；支持 :has() 时卡片边框也变色，不支持时控件与标签仍可用。 */
```

配套文件：[index.html](scripts/02-selectors/index.html)、[index.css](scripts/02-selectors/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/02-selectors/index.html#demo-7)

## 8 伪元素与匹配排错

伪元素（pseudo-element）使用双冒号，描述元素的某个可呈现部分。::before/::after 可生成前后装饰；::first-line 选择排版后的首行，会随可用宽度改变。它们不是 HTML 中新增的真实元素。

本例 content 是生成内容的 CSS 属性，字符串只是装饰；真正的说明保留在 HTML，避免辅助技术漏读重要内容。::before/::after 通常需要 content 产生内容，不用它们给 &lt;img&gt; 等替换元素添加子内容。

排错时先确认样式表加载，再查看实际 DOM 的类名、层级和控件状态；然后区分“选择器有效但当前没匹配”“整条选择器无效”“已经匹配但声明被覆盖”。不要靠把选择器越写越长来处理所有问题。

```html
<p class="generated-note">阅读提醒：先保存进度，再关闭页面。</p>
<p class="first-line-demo">这是用于观察首行的说明文字。缩窄页面后，留意蓝色的文字范围是否随换行改变。</p>
```

```css
.generated-note::before { content: "• "; color: teal; }
.generated-note::after { content: " •"; color: teal; }
.first-line-demo { max-width: 320px; }
.first-line-demo::first-line { color: navy; }
/* 检查 Elements 中伪元素与真实文本的区别；缩窄视口再观察首行。 */
```

配套文件：[index.html](scripts/02-selectors/index.html)、[index.css](scripts/02-selectors/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/02-selectors/index.html#demo-8)

## 本章小结

- 选择器确定匹配对象；类名组合、组合器和逗号列表表达不同的条件关系。
- 伪类匹配状态或结构，伪元素描述可呈现部分；编号依据文档树。
- 普通选择器列表中的无效项可能使整条规则失效；:is()/:where() 的参数容错另有规则。
- 为 :has() 提供可操作的基础页面，为键盘用户保留可见焦点。

## 练习

在 scripts/02-selectors/ 的配套文件中操作，先记录原值，练习后恢复。

（1）只修改 .relations > p 为 .relations p，预测嵌套段落的边框变化；检查容器外段落仍未匹配。

（2）在 .structure-demo 的标题后插入一个 &lt;div&gt;。检查 nth-child(2) 与 nth-of-type(2) 各自指向哪里，再删除新增元素。

（3）用键盘访问 states.html，切换复选框并填写姓名；检查当前选中与有效状态，以及焦点是否始终可辨认。

（4）在开发者工具中临时关闭 @supports 内的边框规则，检查卡片复选框和粗体提示仍工作。

### 提示

计数前先列出父元素的元素子节点；状态题结合 Styles 中匹配的伪类检查，不把访问历史颜色作为自动判断依据。

## 参考与引用来源

- W3C：[Selectors Level 4 §3–6](https://www.w3.org/TR/selectors-4/#structure) 的选择器结构、逻辑组合与属性匹配；[§9](https://www.w3.org/TR/selectors-4/#useraction-pseudos)、[§12](https://www.w3.org/TR/selectors-4/#input-pseudos)、[§13–14](https://www.w3.org/TR/selectors-4/#structural-pseudos) 的交互、结构与组合器；[§4.4](https://www.w3.org/TR/selectors-4/#zero-matches) 的 :where() 优先级。Level 4 为工作草案，按各功能实现范围使用。
- MDN：[Type selectors](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/Type_selectors)、[Class selectors](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/Class_selectors)、[ID selectors](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/ID_selectors) 的基本写法和标识符转义；[Attribute selectors](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/Attribute_selectors#syntax) 的值比较与大小写；[Selector list](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/Selector_list#valid_and_invalid_selector_lists) 的失效与容错；[:nth-child()](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/:nth-child#syntax) 的计数、公式与 of 条件；[:not()](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/:not#description)、[:has()](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/:has#syntax) 的匹配限制及兼容性；[:focus-visible](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/:focus-visible#focus_vs_focus-visible)、[:visited](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/:visited#privacy_restrictions) 的焦点与隐私边界；[Pseudo-elements](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/Pseudo-elements#syntax) 和 [::before](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/::before#description) 的语法与生成内容；[@supports 的选择器检测示例](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@supports#testing_for_the_support_of_a_selector) 的渐进增强条件。
- WHATWG HTML：[The id attribute](https://html.spec.whatwg.org/multipage/dom.html#the-id-attribute) 的文档树内唯一性。